In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Rohini_Delhi_DPCC_2023.xlsx")

In [4]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,320.0,177.0,173.0,99.0,78.0,116.0,51.0,NaN,155.0,150.0,385.0,385.0
1,2,386.0,234.0,217.0,190.0,71.0,102.0,52.0,86.0,168.0,140.0,441.0,371.0
2,3,399.0,241.0,158.0,166.0,138.0,112.0,114.0,NaN,150.0,168.0,482.0,329.0
3,4,352.0,285.0,126.0,93.0,125.0,173.0,134.0,102.0,146.0,178.0,435.0,349.0
4,5,347.0,281.0,132.0,115.0,201.0,153.0,NaN,87.0,118.0,179.0,475.0,308.0
5,6,396.0,303.0,150.0,145.0,258.0,139.0,65.0,102.0,128.0,228.0,458.0,307.0
6,7,384.0,286.0,180.0,119.0,203.0,212.0,68.0,109.0,148.0,286.0,443.0,334.0
7,8,363.0,155.0,219.0,185.0,127.0,161.0,72.0,124.0,94.0,206.0,456.0,357.0
8,9,447.0,239.0,116.0,250.0,212.0,151.0,61.0,132.0,41.0,198.0,453.0,343.0
9,10,399.0,190.0,225.0,200.0,211.0,148.0,NaN,142.0,33.0,NaN,292.0,322.0


In [5]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   33 non-null     float64
 3   March      34 non-null     float64
 4   April      34 non-null     float64
 5   May        36 non-null     float64
 6   June       34 non-null     float64
 7   July       21 non-null     float64
 8   August     32 non-null     float64
 9   September  34 non-null     float64
 10  October    36 non-null     float64
 11  November   34 non-null     float64
 12  December   34 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [6]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [7]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [8]:
# Convert all columns except 'Day' to numeric values
for col in df.columns:
    if col != 'Day':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values with column mean
df_filled = df.fillna(df.mean(numeric_only=True))

In [9]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [10]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,320.0,177.0,173.0,99.0,78.0,116.0,64.285714,112.0625,155.0,150.0,385.0,385.0
1,2,386.0,234.0,217.0,190.0,71.0,102.0,64.285714,86.0000,168.0,140.0,441.0,371.0
2,3,399.0,241.0,158.0,166.0,138.0,112.0,64.285714,112.0625,150.0,168.0,482.0,329.0
3,4,352.0,285.0,126.0,93.0,125.0,173.0,64.285714,102.0000,146.0,178.0,435.0,349.0
4,5,347.0,281.0,132.0,115.0,201.0,153.0,64.285714,87.0000,118.0,179.0,475.0,308.0
